# SCAG Demo

Step-by-step demonstration of the Semantic Coactivation Graph pipeline on M&Ms-2 LAX-4C.

The notebook intentionally exposes every stage instead of calling the full pipeline in one line:

**data → model → activations → X → R → Q → R–Q alignment → graph → communities → semantic analyses → causal ablation**

## 1. Imports

In [1]:
import numpy as np
import torch
from collections import Counter

from scag.config import SCAGConfig
from scag.data import (
    load_mnms2_datasets, build_graph_analysis_cohort,
    make_split_safe, build_loaders, phase_encode_segments,
)
from scag.models import build_model
from scag.activations import extract_activations
from scag.semantics import compute_neuron_concept_matrix
from scag.matrices import compute_R, compute_Q, align_R_Q, compute_edge_reliability
from scag.graph import (
    analyze_graph, density_threshold, seed_stability,
    community_cards, area_coupling,
)
from scag.ablation import community_ablation
from scag.pipeline import compute_area_profiles, run_semantic_granularity

/home/kislay/anaconda3/envs/cinema/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

In [2]:
cfg = SCAGConfig()

print("Device:", cfg.device)
print("Data root:", cfg.data_root)
print("Model root:", cfg.model_root)
print("Classes:", cfg.classes)
print("Concepts:", cfg.concept_names)

Device: cpu
Data root: /media/kislay/New Volume/Turab/results/cinema_reproduction/processed/mnms2
Model root: /media/kislay/New Volume/Turab/results/cinema_reproduction/architecture_bank/2d/mnms2_lax_4c
Classes: ('ARR', 'CIA', 'FALL', 'HCM', 'LV', 'NOR')
Concepts: ('background', 'RV_ED', 'MYO_ED', 'LV_ED', 'RV_ES', 'MYO_ES', 'LV_ES')


/home/kislay/anaconda3/envs/cinema/lib/python3.11/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


## 3. Load M&Ms-2 LAX-4C datasets

In [3]:
train_dataset, val_dataset, test_dataset, splits = load_mnms2_datasets(cfg)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Loaded MnMs2 LAX-4C train patients: 160
Loaded MnMs2 LAX-4C val patients: 30
Loaded MnMs2 LAX-4C test patients: 110
Anatomy concepts: {0: 'background', 1: 'RV', 2: 'MYO', 3: 'LV'}
Train: 160
Val: 30
Test: 110


## 4. Build the graph-analysis cohort

In [4]:
eval_dataset = build_graph_analysis_cohort(val_dataset, test_dataset)
eval_dataset = make_split_safe(train_dataset, eval_dataset)

print("Graph-analysis cohort:", len(eval_dataset))

[SCAG] Graph cohort: val (30) + test (110) = 140 patients.
[SCAG] Official test set is NOT held out in this graph analysis.
[split-safe] train source=images, eval source=images
[split-safe] 140 -> 140 (dropped 0 train-overlapping)
Graph-analysis cohort: 140


## 5. Build loaders

In [5]:
train_loader, eval_loader = build_loaders(
    train_dataset,
    eval_dataset,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
)

print("Train batches:", len(train_loader))
print("Evaluation batches:", len(eval_loader))

Train batches: 10
Evaluation batches: 9


## 6. Load one trained model

In [6]:
tag = "resnet18"

model, hook_target, layer_name = build_model(tag, cfg)
model = model.to(cfg.device).eval()

print("Model:", tag)
print("Feature layer:", layer_name)

[resnet18] loaded checkpoint: /media/kislay/New Volume/Turab/results/cinema_reproduction/architecture_bank/2d/mnms2_lax_4c/resnet18/stacked/imagenet/seed_0/best_val_mcc.pt
Model: resnet18
Feature layer: encoder.layer4


## 7. Extract training activations

In [7]:
train_acts, train_segs = extract_activations(
    model, train_loader, hook_target, cfg.device
)

print("Activations:", train_acts.shape)
print("Masks:", train_segs.shape)

Activations: torch.Size([160, 512, 8, 8])
Masks: torch.Size([160, 2, 256, 256])


## 8. Extract evaluation activations

In [8]:
eval_acts, eval_segs = extract_activations(
    model, eval_loader, hook_target, cfg.device
)

print("Activations:", eval_acts.shape)
print("Masks:", eval_segs.shape)

Activations: torch.Size([140, 512, 8, 8])
Masks: torch.Size([140, 2, 256, 256])


## 9. Encode ED/ES-specific concepts

In [9]:
train_segs = phase_encode_segments(train_segs)
eval_segs = phase_encode_segments(eval_segs)

print("Encoded labels:", torch.unique(train_segs).tolist())

Encoded labels: [0, 1, 2, 3, 4, 5, 6]


## 10. Compute neuron–concept associations X

`X` has one row per neuron/filter and one column per phase-aware concept:

`background, RV_ED, MYO_ED, LV_ED, RV_ES, MYO_ES, LV_ES`.

In [10]:
X, labeled = compute_neuron_concept_matrix(
    train_acts,
    train_segs,
    num_concepts=cfg.num_concepts,
    alpha=cfg.alpha,
    eta=cfg.eta,
)

lab_idx = np.where(labeled)[0]

print("X:", X.shape)
print("Labeled neurons:", len(lab_idx), "/", X.shape[0])

  dissect 128/512
  dissect 256/512
  dissect 384/512
  dissect 512/512
X: (512, 7)
Labeled neurons: 313 / 512


## 11. Inspect dominant neuron concepts

In [11]:
counts = Counter(X[lab_idx].argmax(axis=1).tolist())
{
    cfg.concept_name_map[int(concept)]: int(count)
    for concept, count in sorted(counts.items())
}

{'RV_ED': 1,
 'MYO_ED': 10,
 'LV_ED': 48,
 'RV_ES': 10,
 'MYO_ES': 12,
 'LV_ES': 232}

## 12. Compute the coactivation matrix R

Activation maps are spatially pooled and Spearman correlation is computed between neurons.

In [12]:
R_train = compute_R(train_acts)
R_eval = compute_R(eval_acts)

print("R_train:", R_train.shape)
print("R_eval:", R_eval.shape)

R_train: (512, 512)
R_eval: (512, 512)


## 13. Compute semantic similarity Q

`Q` is cosine similarity between the concept profiles of the labeled neurons.

In [13]:
Q = compute_Q(X[lab_idx])

print("Q:", Q.shape)

Q: (313, 313)


## 14. Train–evaluation edge reliability

In [14]:
edge_reliability = compute_edge_reliability(R_train, R_eval, lab_idx)
print(f"rho = {edge_reliability:.4f}")

rho = 0.8663


## 15. Align R and Q

In [15]:
R_train_lab = R_train[np.ix_(lab_idx, lab_idx)]
R_eval_lab = R_eval[np.ix_(lab_idx, lab_idx)]

alignment_train = align_R_Q(
    R_train_lab, Q, n_perm=cfg.n_mantel_permutations
)
alignment_eval = align_R_Q(
    R_eval_lab, Q, n_perm=cfg.n_mantel_permutations
)

print("Train (r, z, p):", alignment_train)
print("Eval  (r, z, p):", alignment_eval)

Train (r, z, p): (0.24116687339085632, 23.616349445503108, 0.000999000999000999)
Eval  (r, z, p): (0.2173708968094293, 15.771915433965955, 0.000999000999000999)


## 16. Analyze the fixed-threshold coactivation graph

In [16]:
graph_eval, artifacts_eval = analyze_graph(
    R_eval,
    X,
    labeled,
    tau=cfg.tau,
    n_perm=cfg.n_battery_permutations,
)

graph_eval

{'n_nodes': 512,
 'n_edges': 30803,
 'n_isolates': 0,
 'n_comm': 4,
 'modularity': 0.5029021528932908,
 'purity': 0.7225122708920713,
 'purity_z': -2.5125448945594235,
 'purity_p': 0.9940119760479041,
 'ami': 0.04499773063054178,
 'ami_z': 5.5219324043291715,
 'ami_p': 0.001996007984031936,
 'assort': 0.04901591688889767,
 'assort_z': 11.627254377336508,
 'assort_p': 0.001996007984031936,
 'soft_delta': 0.019491788933656218,
 'soft_z': 5.989738468404039,
 'soft_p': 0.001996007984031936,
 'alignment_tau_r': 0.21463350053785837,
 'alignment_tau_z': 18.732743781524064,
 'alignment_tau_p': 0.001996007984031936}

## 17. Inspect Louvain communities

In [17]:
communities = artifacts_eval["communities"]

print("Communities:", len(communities))
for i, community in enumerate(communities):
    print(f"Community {i:02d}: {len(community)} neurons")

Communities: 4
Community 00: 165 neurons
Community 01: 145 neurons
Community 02: 113 neurons
Community 03: 89 neurons


## 18. Analyze a density-matched graph

In [18]:
tau_dm = density_threshold(R_eval, cfg.graph_density)

graph_dm, artifacts_dm = analyze_graph(
    R_eval,
    X,
    labeled,
    tau=tau_dm,
    n_perm=cfg.n_battery_permutations,
)

print("tau*:", tau_dm)
print("Edges:", graph_dm["n_edges"])
print("Modularity:", graph_dm["modularity"])

tau*: 0.888160956198738
Edges: 2617
Modularity: 0.8052996597315413


## 19. Compare anatomy, phase, and anatomy × phase

In [19]:
granularity_df, granularity_summary, semantic_views = run_semantic_granularity(
    tag=tag,
    R_train=R_train,
    R_eval=R_eval,
    X=X,
    labeled=labeled,
    tau_dm=tau_dm,
    cfg=cfg,
)

granularity_df

  comm 00: RV[anatomy]x3, MYO[anatomy]x6, LV[anatomy]x62
  comm 01: LV[anatomy]x107, MYO[anatomy]x5, RV[anatomy]x1
  comm 02: MYO[anatomy]x11, RV[anatomy]x4, LV[anatomy]x64
  comm 03: LV[anatomy]x47, MYO[anatomy]x2, RV[anatomy]x1
  comm 00: ED[phase]x20, ES[phase]x51
  comm 01: ES[phase]x106, ED[phase]x7
  comm 02: ES[phase]x74, ED[phase]x5
  comm 03: ES[phase]x44, ED[phase]x6
  comm 00: LV_ED[anatomy_phase]x21, MYO_ED[anatomy_phase]x6, RV_ED[anatomy_phase]x1, LV_ES[anatomy_phase]x41, RV_ES[anatomy_phase]x2
  comm 01: LV_ES[anatomy_phase]x97, LV_ED[anatomy_phase]x9, RV_ES[anatomy_phase]x1, MYO_ES[anatomy_phase]x4, MYO_ED[anatomy_phase]x2
  comm 02: MYO_ES[anatomy_phase]x7, RV_ES[anatomy_phase]x6, LV_ES[anatomy_phase]x58, LV_ED[anatomy_phase]x7, MYO_ED[anatomy_phase]x1
  comm 03: LV_ED[anatomy_phase]x11, LV_ES[anatomy_phase]x36, MYO_ES[anatomy_phase]x1, RV_ES[anatomy_phase]x1, MYO_ED[anatomy_phase]x1


,tag,semantic_level,n_labeled,n_effective_classes,dominant_counts,alignment_train_r,alignment_train_z,alignment_train_p,alignment_eval_r,alignment_eval_z,...,eval_assort_z,eval_soft_delta,eval_soft_z,eval_alignment_tau_r,eval_alignment_tau_z,eval_modularity,dm_ami_z,dm_assort_z,dm_soft_z,dm_modularity
0,resnet18,anatomy,313,3,"{""RV"": 9, ""MYO"": 24, ""LV"": 280}",0.226839,22.583258,0.000999,0.199649,14.731560,...,8.311981,0.016769,5.264317,0.197153,17.372711,0.502902,3.352002,5.141159,6.614512,0.8053
1,resnet18,phase,313,2,"{""ED"": 38, ""ES"": 275}",0.237660,23.100622,0.000999,0.254927,19.419118,...,10.168679,0.003294,8.495131,0.232731,21.416862,0.502902,6.177106,5.695623,6.483311,0.8053
2,resnet18,anatomy_phase,313,6,"{""RV_ED"": 1, ""MYO_ED"": 10, ""LV_ED"": 48, ""RV_ES...",0.241167,23.616349,0.000999,0.217371,15.771915,...,11.627254,0.019492,5.989738,0.214634,18.732744,0.502902,6.051110,9.247330,7.304846,0.8053


## 20. Louvain seed stability

In [20]:
ami_mean, ami_sd, partition_stability = seed_stability(
    artifacts_eval["G"],
    artifacts_eval["lab_nodes"],
    artifacts_eval["labels"],
    n_seeds=cfg.n_louvain_seeds,
)

print("AMI mean:", ami_mean)
print("AMI SD:", ami_sd)
print("Partition stability:", partition_stability)

  seed  0: AMI=0.0457 z_null=+5.8
  seed  1: AMI=0.0457 z_null=+5.7
  seed  2: AMI=0.0457 z_null=+5.8
  seed  3: AMI=0.0407 z_null=+5.1
  seed  4: AMI=0.0419 z_null=+5.4
  seed  5: AMI=0.0419 z_null=+5.4
  seed  6: AMI=0.0462 z_null=+5.8
  seed  7: AMI=0.0407 z_null=+5.0
  seed  8: AMI=0.0457 z_null=+5.8
  seed  9: AMI=0.0457 z_null=+6.1
AMI mean: 0.04395838506203562
AMI SD: 0.0022136466386789397
Partition stability: 0.9324785400504367


## 21. Community semantic enrichment

In [21]:
enrichment_df = community_cards(
    artifacts_eval,
    cfg.concept_name_map,
    cfg.concept_category_map,
)

enrichment_df

  comm 00: LV_ED[anatomy_phase]x21, MYO_ED[anatomy_phase]x6, RV_ED[anatomy_phase]x1, LV_ES[anatomy_phase]x41, RV_ES[anatomy_phase]x2
  comm 01: LV_ES[anatomy_phase]x97, LV_ED[anatomy_phase]x9, RV_ES[anatomy_phase]x1, MYO_ES[anatomy_phase]x4, MYO_ED[anatomy_phase]x2
  comm 02: MYO_ES[anatomy_phase]x7, RV_ES[anatomy_phase]x6, LV_ES[anatomy_phase]x58, LV_ED[anatomy_phase]x7, MYO_ED[anatomy_phase]x1
  comm 03: LV_ED[anatomy_phase]x11, LV_ES[anatomy_phase]x36, MYO_ES[anatomy_phase]x1, RV_ES[anatomy_phase]x1, MYO_ED[anatomy_phase]x1


,comm,concept,k,p,p_adj
0,0,6,41,0.999851,0.999851
1,0,2,6,0.010877,0.066686
2,0,3,21,0.000322,0.003220
3,0,1,1,0.226837,0.648106
4,0,4,2,0.704123,0.999851
5,1,6,97,0.000212,0.003220
6,1,3,9,0.998679,0.999851
7,1,4,1,0.989562,0.999851
8,1,5,4,0.686644,0.999851
9,1,2,2,0.927805,0.999851


## 22. Anatomy-area coupling

In [22]:
area_profiles = compute_area_profiles(train_segs, cfg.num_concepts)

area_stats = area_coupling(
    R_eval,
    X,
    labeled,
    area_profiles,
    n_perm=cfg.n_cooccurrence_permutations,
)

area_stats

  positive (n=14358): area-profile cosine=0.9855 z=+5.8 p=0.0010
      anti (n= 7961): area-profile cosine=0.9799 z=-1.1 p=0.1399


{'positive': (14358,
  0.9855270841186757,
  5.752455997392664,
  0.000999000999000999),
 'anti': (7961, 0.979909620258056, -1.0576679487886642, 0.13986013986013987)}

## 23. Causal community ablation

Each Louvain community is ablated by zeroing its channels at the selected feature layer. Its prediction effect is compared with size-matched random neuron ablations.

In [23]:
node2label = dict(zip(
    artifacts_eval["lab_nodes"],
    artifacts_eval["labels"].tolist(),
))

ablation_df = community_ablation(
    model=model,
    hook_target=hook_target,
    loader=eval_loader,
    communities=artifacts_eval["communities"],
    num_filters=train_acts.shape[1],
    device=cfg.device,
    val_segs=eval_segs,
    node2label=node2label,
    concept_names=cfg.concept_name_map,
    n_random=cfg.n_ablation_samples,
)

ablation_df

KeyboardInterrupt: 

## 24. Rank communities by ablation effect

In [ ]:
if ablation_df is not None and len(ablation_df):
    display(
        ablation_df[
            ["comm", "size", "kl", "rnd", "rnd_sd", "flip"]
        ].sort_values("kl", ascending=False)
    )

## Outputs now available

- `X`: neuron–concept association matrix
- `R_train`, `R_eval`: functional coactivation matrices
- `Q`: semantic similarity matrix
- `alignment_train`, `alignment_eval`: full-matrix R–Q alignment
- `artifacts_eval["G"]`: NetworkX coactivation graph
- `artifacts_eval["communities"]`: Louvain communities
- `granularity_df`: anatomy vs phase vs anatomy×phase comparison
- `enrichment_df`: community semantic enrichment
- `area_stats`: anatomy-area coupling
- `ablation_df`: causal community-ablation results

For the automated multi-architecture experiment, use `python run_scag.py`.